# Notebook 01: Generate Preference Data for DPO

This notebook generates preference pairs (chosen/rejected) for DPO training using two methods:
1. **Synthetic mode**: DeepSeek generates both correct and intentionally flawed responses
2. **On-policy mode**: SFT model samples candidates, DeepSeek judges and ranks them

**Requirements**: DeepSeek API key (set as environment variable)

**GPU**: Not needed for synthetic mode. L4 needed for on-policy mode (SFT model sampling).

## Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Clone or copy project
import os
PROJECT_DIR = '/content/smallllm-dpo'
DRIVE_DIR = '/content/drive/MyDrive/smallllm-dpo'

if os.path.exists(DRIVE_DIR):
    !cp -r {DRIVE_DIR} {PROJECT_DIR}
    print(f'Copied from Drive: {DRIVE_DIR}')
else:
    !git clone https://github.com/XIECHENG6/smallllm-dpo.git {PROJECT_DIR}
    print('Cloned from GitHub')

os.chdir(PROJECT_DIR)
!pip install -q openai tqdm datasets pyyaml

In [ ]:
# Set DeepSeek API key
import os
from google.colab import userdata

# Option 1: From Colab secrets
try:
    os.environ['DEEPSEEK_API_KEY'] = userdata.get('DEEPSEEK_API_KEY')
    print('API key loaded from Colab secrets')
except:
    # Option 2: Manual input
    import getpass
    os.environ['DEEPSEEK_API_KEY'] = getpass.getpass('Enter DeepSeek API key: ')

In [ ]:
import sys
sys.path.insert(0, PROJECT_DIR)

from src.utils.llm_client import LLMClient
from src.data.generator import ScenarioGenerator
from src.data.formatter import pairs_to_dataset, save_pairs_jsonl, dataset_statistics
from src.data.function_pool import FUNCTION_POOL, DOMAIN_MAP

client = LLMClient()

# Quick connectivity test
resp = client.chat([{'role': 'user', 'content': 'Say OK'}], max_tokens=5)
print(f'DeepSeek API connected: {resp}')

## Part 1: Generate Scenarios

Use DeepSeek to generate diverse function calling scenarios across 25 functions and 5 domains.

In [ ]:
generator = ScenarioGenerator(client, seed=42)

# Generate 1000 scenarios (adjust as needed)
NUM_SCENARIOS = 1000
scenarios = generator.generate_scenarios(NUM_SCENARIOS, batch_size=10)
print(f'Generated {len(scenarios)} scenarios')

# Show a few examples
for s in scenarios[:3]:
    print(f"\nQuery: {s['query']}")
    print(f"Function: {s['expected_function']}")
    print(f"Args: {s['expected_arguments']}")

In [ ]:
# Check scenario diversity
from collections import Counter
func_dist = Counter(s['expected_function'] for s in scenarios)
print('Function distribution:')
for fn, count in func_dist.most_common():
    print(f'  {fn}: {count}')

## Part 2: Synthetic Preference Pairs

For each scenario, DeepSeek generates a rejected response with a specific error type.
The chosen response is the correct function call from the scenario.

In [ ]:
synthetic_pairs = generator.build_preference_pairs(scenarios)
print(f'Built {len(synthetic_pairs)} synthetic preference pairs')

# Statistics
stats = dataset_statistics(synthetic_pairs)
print(f"\nError type distribution:")
for et, count in stats['error_type_distribution'].items():
    print(f'  {et}: {count}')
print(f"\nAvg chosen length: {stats['avg_chosen_length']:.0f} chars")
print(f"Avg rejected length: {stats['avg_rejected_length']:.0f} chars")

In [ ]:
# Inspect a few pairs
for pair in synthetic_pairs[:3]:
    print(f"\nQuery: {pair['query']}")
    print(f"Error type: {pair['error_type']}")
    print(f"Chosen:   {pair['chosen'][:100]}")
    print(f"Rejected: {pair['rejected'][:100]}")

## Part 3: On-Policy Preference Pairs (Optional, requires GPU)

Load the SFT model from Phase 1, sample multiple responses per scenario,
then use DeepSeek-as-Judge to rank and create preference pairs.

In [ ]:
ON_POLICY = False  # Set to True if running on GPU and have SFT adapter

if ON_POLICY:
    !pip install -q torch transformers peft bitsandbytes accelerate
    
    from src.data.sampler import SFTSampler
    from src.data.judge import ResponseJudge
    
    # Load SFT model — update adapter path to your HuggingFace model
    sampler = SFTSampler(
        base_model_name='Qwen/Qwen2.5-3B-Instruct',
        adapter_path='Cheng-1/qwen2.5-3b-fc-lora',  # Update this
    )
    
    # Sample from a subset of scenarios
    on_policy_scenarios = scenarios[:200]
    sampled = sampler.sample_batch(on_policy_scenarios, n_per_scenario=4, temperature=0.8)
    
    # Judge and rank
    judge = ResponseJudge(client)
    on_policy_pairs = judge.build_pairs_from_samples(sampled, min_score_gap=3)
    print(f'Built {len(on_policy_pairs)} on-policy preference pairs')
else:
    on_policy_pairs = []
    print('Skipping on-policy mode (set ON_POLICY=True to enable)')

## Part 4: Save Dataset

In [ ]:
from src.data.formatter import merge_pair_sources

# Merge all sources
all_pairs = merge_pair_sources(synthetic_pairs, on_policy_pairs)

# Save as JSONL
os.makedirs('data', exist_ok=True)
save_pairs_jsonl(all_pairs, 'data/preference_pairs.jsonl')

# Convert to HuggingFace Dataset
train_ds, test_ds = pairs_to_dataset(all_pairs, train_ratio=0.9, seed=42)
print(f'\nTrain: {len(train_ds)} samples')
print(f'Test:  {len(test_ds)} samples')

# Save datasets
train_ds.save_to_disk('data/train_dataset')
test_ds.save_to_disk('data/test_dataset')
print('Datasets saved to data/')

In [ ]:
# Save scenarios for evaluation
import json
save_pairs_jsonl(scenarios, 'data/scenarios.jsonl')

# Copy to Drive for persistence
!mkdir -p {DRIVE_DIR}/data
!cp data/preference_pairs.jsonl {DRIVE_DIR}/data/
!cp -r data/train_dataset {DRIVE_DIR}/data/
!cp -r data/test_dataset {DRIVE_DIR}/data/
!cp data/scenarios.jsonl {DRIVE_DIR}/data/
print(f'Data saved to Drive: {DRIVE_DIR}/data/')

In [ ]:
# API usage summary
print('DeepSeek API Usage:')
for k, v in client.usage_summary.items():
    print(f'  {k}: {v:,}')

# Rough cost estimate (DeepSeek V3 pricing)
input_cost = client.total_input_tokens / 1_000_000 * 1  # ~1 CNY / M tokens
output_cost = client.total_output_tokens / 1_000_000 * 2  # ~2 CNY / M tokens
print(f'\nEstimated cost: ~{input_cost + output_cost:.2f} CNY')